# IDM-VTON Cloud Worker API
This notebook installs and runs the IDM-VTON virtual try-on engine on a free Google Colab GPU. 
Once running, it provides a public `gradio.live` link that you can paste into your local Virtual Try-On app to bypass usage limits!

**Instructions:**
1. Go to the top menu and select **Runtime > Change runtime type**.
2. Ensure the **Hardware accelerator** is set to **T4 GPU**.
3. Run the setup cell below (Play button). It will take about ~3-5 minutes to install everything.
4. Run the API cell. It will print a `Running on public URL: https://xxxx.gradio.live` link.
5. Copy that link and paste it into your local app's settings!

In [ ]:
# Step 1: Clone repo and install dependencies
!git clone https://github.com/yisol/IDM-VTON.git
%cd /content/IDM-VTON
!pip install diffusers==0.25.0 accelerate==0.25.0 gradio>=4.30.0 gradio_client>=0.16.0 transformers==4.36.2 einops==0.7.0 bitsandbytes==0.39.0 fvcore cloudpickle omegaconf pycocotools basicsr av onnxruntime huggingface-hub==0.22.2

In [ ]:
# Step 2: Start the API Server
import os
%cd /content/IDM-VTON

# We patch the app.py file to ensure the Gradio app runs with share=True so it's accessible from the outside.
with open('gradio_demo/app.py', 'r') as f:
    content = f.read()
    
if 'image_blocks.launch(share=True' not in content:
    content = content.replace('image_blocks.launch()', 'image_blocks.launch(share=True)')
    
if 'low_cpu_mem_usage=True' not in content:
    content = content.replace('torch_dtype=torch.float16,', 'torch_dtype=torch.float16, low_cpu_mem_usage=True,')

with open('gradio_demo/app.py', 'w') as f:
    f.write(content)

if "base_path = 'yisol/IDM-VTON'" in content:
    content = content.replace("base_path = 'yisol/IDM-VTON'", "base_path = 'camenduru/IDM-VTON-F16'")

print("Starting IDM-VTON API... Please wait while models are downloaded and loaded into the GPU.")
!python gradio_demo/app.py